# Extract Swedish words from Youtube with musick and effect as background

When learning Swedish independently, children's educational videos are actually every effective in my case. However, children's videos often contains fun musick and effect in the background. 

The downloaded mp3 does not actually contain separate “speech” and “background” layers anymore. They were mixed together when the video was produced. But we can use a source-separation model to estimate:

```text
original audio
      ↓
 ┌────────────┐
 │            │
vocals     background
speech     music + effects
````

## 1. Install packages

```bash
pip install demucs
demucs --two-stems=vocals alphabet_audio.mp3
```

This creates something like:
separated/
└── htdemucs/
    └── alphabet_audio/
        ├── vocals.wav
        └── no_vocals.wav

## Use the KB-Whisper model

We then can use the Swedish tailored whiper model to extract the Swedish examples in `vocals.wav`.

In [16]:
from pathlib import Path
from transformers import pipeline

transcriber = pipeline(
    task="automatic-speech-recognition",
    model="KBLab/kb-whisper-large",
)

clean_audio_path = Path(
    "separated/htdemucs/alphabet_audio/vocals.wav"
)

result_clean = transcriber(
    str(clean_audio_path),
    generate_kwargs={
        "language": "sv",
        "task": "transcribe",
    },
    return_timestamps=True,
)

print(result_clean["text"])

config.json:   0%|          | 0.00/3.72k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.85k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

 Banan och avokado! Hej allihopa! Idag ska vi titta på alla bokstäver i alfabetet och vilka ord man kan använda bokstäverna i. Häng med! Alfabetet börjar med bokstaven A. Så här ser stora A ut. Och så här ser lilla A ut. Här kommer några ord som börjar. A som i anka. A som i apelsin. A som i apa. A som i ankare. A som i ananas. A som i abborre. Bokstaven B. Stora B. Och lilla B. B som i båt. B som i bam. Banan. B som i bil. B som i bok. B som i broccoli. Efter B så kommer bokstaven C. Stora C och lilla C. C som i cykel. C som i citron. C som i clementin. C som i champinjon. C som i choklad. Bokstaven D. Stora D och lilla D. D som i daggmask. D som i... DELFIN D som i dörr. D som i DRAGKEDJA D som i DATOR E Stora E och Lilla E. E som i ekorre. E som i eld. E som i elefant. E som i enhörning. Efter E kommer bokstaven F. Stora F och lilla f. F som i får. F som i flaska. i fågel. F som i fjäril. F som i fjäder. Sen har vi bokstaven G. Stora G och lilla G. G som i gran. G som i get. G som i

In [17]:
import pandas as pd

transcript_rows = []

for chunk in result_clean["chunks"]:
    start_time, end_time = chunk["timestamp"]

    transcript_rows.append(
        {
            "start": start_time,
            "end": end_time,
            "text": chunk["text"].strip(),
        }
    )

transcript_df = pd.DataFrame(transcript_rows)
transcript_df.head(20)


,start,end,text
0,0.0,3.0,Banan och avokado!
1,7.0,9.0,Hej allihopa!
2,9.0,13.0,Idag ska vi titta på alla bokstäver i alfabetet
3,13.0,17.0,och vilka ord man kan använda bokstäverna i.
4,17.0,19.0,Häng med!
5,19.0,23.0,Alfabetet börjar med bokstaven A.
6,23.0,25.0,Så här ser stora A ut.
7,25.0,28.0,Och så här ser lilla A ut.
8,28.0,30.0,Här kommer några ord som börjar.
9,30.0,35.0,A som i anka.


In [20]:
import re

transcript_text = result_clean["text"]

pattern = re.compile(
    r"\bsom\s+i\b[\s,.:;…-]*"
    r"([A-Za-zÅÄÖåäö]+(?:-[A-Za-zÅÄÖåäö]+)*)",
    flags=re.IGNORECASE,
)

raw_words = pattern.findall(transcript_text)

candidate_df = pd.DataFrame({"raw_word": raw_words})

candidate_df["letter"] = candidate_df["raw_word"].str[0].str.upper()

candidate_df["word"] = candidate_df["raw_word"].str.strip().str.lower()

print("Number of words found:", candidate_df["word"].nunique())
print(candidate_df[candidate_df["word"].duplicated(keep=False)])

candidate_df = candidate_df[~candidate_df["word"].duplicated(keep=False)]

candidate_df.head(20)


Number of words found: 119
Empty DataFrame
Columns: [raw_word, letter, word]
Index: []


,raw_word,letter,word
0,anka,A,anka
1,apelsin,A,apelsin
2,apa,A,apa
3,ankare,A,ankare
4,ananas,A,ananas
5,abborre,A,abborre
6,båt,B,båt
7,bam,B,bam
8,bil,B,bil
9,bok,B,bok


In [18]:
transcript_text = result_clean["text"]

In [19]:
from typing import Literal
from pydantic import BaseModel, Field


class AlphabetExample(BaseModel):
    letter: str = Field(
        description="The corresponding Swedish alphabet letter."
    )
    
    word: str = Field(
        description="The normalized standard Swedish noun."
    )
    confidence: Literal["high", "medium", "low"]
    source_text: str = Field(
        description="Nearby original transcript text supporting the extraction."
    )


class AlphabetExtraction(BaseModel):
    examples: list[AlphabetExample]


In [14]:
from openai import OpenAI

client = OpenAI()

response = client.responses.parse(
    model="gpt-5.2",
    reasoning={"effort": "low"},
    input=[
        {
            "role": "system",
            "content": """
                        Extract the Swedish example words from the
                        speech-recognition transcript according to the swedish alphabet.

                        Desired output:
                            - letter		
                            - word	
                            - confidence
                            - source_text

                        Each letter is followeed by several examples words starting with that letter. 
                        The examples are read aloud in Swedish, and the transcript may contain misheard or misrecognized words.
                        Remember that continuous several examples should start from the same letter.

                        Do not invent examples if you can not infer them from the transcript.
                        """,
        },
        {
            "role": "user",
            "content": transcript_text,
        },
    ],
    text_format=AlphabetExtraction,
)

api_result = response.output_parsed


In [ ]:
api_review_df = pd.DataFrame(
    example.model_dump()
    for example in api_result.examples
)

print("Rows returned:", len(api_review_df))
api_review_df.iloc[105:110, :]


Rows returned: 125


,letter,word
105,W,wok
106,X,xylofon
107,Y,yxa
108,Y,yoga
109,Z,zucchini


In [ ]:
uncertain_rows = api_review_df.loc[
    api_review_df["confidence"] != "high"
]

print("Uncertain rows:", len(uncertain_rows))

uncertain_rows.head(20)

In [12]:
api_swedish_df = api_review_df[["letter", "word"]]

print("Letters found:", api_swedish_df["letter"].unique())
print("Swedish words:", ", ".join(api_swedish_df["word"]))

Letters found: <StringArray>
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O',
 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'Å', 'Ä', 'Ö']
Length: 29, dtype: str
Swedish words: anka, apelsin, apa, ankare, ananas, abborre, båt, banan, bil, bok, broccoli, cykel, citron, clementin, champinjon, choklad, daggmask, delfin, dörr, dragkedja, dator, ekorre, eld, elefant, enhörning, får, flaska, fågel, fjäril, fjäder, gran, get, gaffel, gem, gurka, hare, haj, handduk, hammare, handväska, insekt, istapp, igelkott, iglo, jacka, juice, julgran, jordgubbe, jultomten, kiwi, katt, kaffe, kalv, kaktus, lampa, lamm, lakrits, lego, måne, mus, maskros, morot, makaroner, nyckel, napp, nalle, nektarin, nypon, overall, oliv, oxe, orm, päron, planet, palm, pyjamas, pirat, quinoa, räv, rabarber, ring, robot, regnbåge, solveig, sallad, snöflinga, sax, säl, tax, tejp, tand, tofflor, tandkräm, u-båt, ur, uggla, utomjording, vulkan, vatten, vas, vanilj, val, wienerbröd, walkie-